# 📈 SBER 5-Minute Intraday Research Pipeline

**Goal:** feature engineering on raw Finam OHLCV candles → supervised ML model → probability calibration.

**Target variable:** `target_is_green_next = 1` if next 5m candle closes above its open, else 0.

**Important:** all features use only current-bar-close or past-bar data. No look-ahead bias.

## Cell 1 — Imports & Display Settings

In [ ]:
# ── Стандартные импорты ────────────────────────────────────────────────────
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pandas import DataFrame, concat

# ── Технические индикаторы ─────────────────────────────────────────────────
import pandas_ta as ta

# ── Визуализация ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (15, 5)

# ── Machine Learning ───────────────────────────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    accuracy_score, roc_auc_score, log_loss,
    brier_score_loss, classification_report, confusion_matrix,
)

print('Python:', sys.version)
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('All imports OK ✓')

## Cell 2 — Helper: series_to_supervised

In [ ]:
# ── series_to_supervised ──────────────────────────────────────────────────
# Преобразует многомерный временной ряд в датасет для обучения с учителем.
# Источник: https://machinelearningmastery.com/convert-time-series-supervised-learning-problem-python/
# Адаптировано: используем имена колонок из исходного DataFrame вместо var1, var2 ...

def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
    '''
    Frame a time series as a supervised learning dataset.
    Arguments:
        data: DataFrame — многомерный временной ряд.
        n_in: int — количество лаговых наблюдений (X).
        n_out: int — количество будущих наблюдений (y).
        dropnan: bool — удалять ли строки с NaN.
    Returns:
        pd.DataFrame — датасет для supervised learning.
    '''
    n_vars = 1 if type(data) is list else data.shape[1]
    df = DataFrame(data)
    col_list = data.columns.tolist()  # имена колонок для человекочитаемых меток
    cols, names = list(), list()

    # Лаговые входные колонки: t-n_in, ..., t-1
    for i in range(n_in, 0, -1):
        cols.append(df.shift(i))
        names += [('%s_%d(t-%d)' % (col_list[j], j+1, i)) for j in range(n_vars)]

    # Выходные колонки прогноза: t, t+1, ..., t+n_out-1
    for i in range(0, n_out):
        cols.append(df.shift(-i))
        if i == 0:
            names += [('%s_var%d(t)' % (col_list[j], j+1)) for j in range(n_vars)]
        else:
            names += [('%s_var%d(t+%d)' % (col_list[j], j+1, i)) for j in range(n_vars)]

    # Собираем всё в один DataFrame
    agg = concat(cols, axis=1)
    agg.columns = names
    if dropnan:
        agg.dropna(inplace=True)
    return agg

print('series_to_supervised defined ✓')

## Cell 3 — Feature Engineering Functions

In [ ]:
# ── Константы ─────────────────────────────────────────────────────────────
EPS = 1e-12  # Защита от деления на ноль у 'плоских' свечей (HIGH == LOW)
DATA_PATH = './data/Сбербанк/year_result.csv'  # Путь к исходным данным


# ── 1. Подготовка OHLCV ───────────────────────────────────────────────────
def prepare_ohlcv_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Валидирует колонки, приводит типы, парсит datetime, сортирует по времени.
    Входной DataFrame не мутируется — возвращается копия.
    '''
    required = {'OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOL'}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    out = df.copy(deep=True)
    for col in ['OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOL']:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    if {'DATE', 'TIME'}.issubset(out.columns):
        date_str = out['DATE'].astype(str).str.strip()
        time_str = out['TIME'].astype(str).str.zfill(6).str.strip()
        out['datetime'] = pd.to_datetime(date_str + time_str, format='%Y%m%d%H%M%S', errors='coerce')
        out = out.sort_values('datetime', kind='stable').reset_index(drop=True)
    return out


# ── 2. Геометрия свечи ────────────────────────────────────────────────────
def add_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Добавляет признаки геометрии свечи и однобарные returns.
    Использует только текущий и предыдущий бар — без look-ahead bias.
    '''
    out = df.copy(deep=True)
    rng = out['HIGH'] - out['LOW']
    safe_rng = rng.where(rng.abs() > EPS, np.nan)
    max_oc = np.maximum(out['OPEN'], out['CLOSE'])
    min_oc = np.minimum(out['OPEN'], out['CLOSE'])
    prev_close = out['CLOSE'].shift(1)
    prev_high  = out['HIGH'].shift(1)
    prev_low   = out['LOW'].shift(1)
    # Тело и тени свечи
    out['candle_body']         = out['CLOSE'] - out['OPEN']
    out['body_abs']            = out['candle_body'].abs()
    out['candle_range']        = rng
    out['upper_wick']          = out['HIGH'] - max_oc
    out['lower_wick']          = min_oc - out['LOW']
    # Относительные соотношения внутри свечи
    out['body_to_range']       = (out['body_abs'] / safe_rng).clip(0, 1)
    out['upper_wick_to_range'] = (out['upper_wick'] / safe_rng).clip(0, 1)
    out['lower_wick_to_range'] = (out['lower_wick'] / safe_rng).clip(0, 1)
    out['close_pos_in_range']  = ((out['CLOSE'] - out['LOW']) / safe_rng).clip(0, 1)
    out['open_pos_in_range']   = ((out['OPEN'] - out['LOW']) / safe_rng).clip(0, 1)
    # Направление и тип свечи
    out['direction']           = np.sign(out['candle_body']).astype('float64')
    out['is_green']            = (out['CLOSE'] > out['OPEN']).astype('int8')
    out['is_red']              = (out['CLOSE'] < out['OPEN']).astype('int8')
    out['is_doji_like']        = (out['body_to_range'] <= 0.1).astype('int8')
    # Returns и gap
    out['ret_1']               = out['CLOSE'].pct_change(1)
    out['gap_from_prev_close'] = out['OPEN'] / prev_close - 1.0
    out['close_to_prev_high']  = out['CLOSE'] / prev_high - 1.0
    out['close_to_prev_low']   = out['CLOSE'] / prev_low - 1.0
    out['range_pct_close']     = out['candle_range'] / out['CLOSE'].replace(0, np.nan)
    # Volume features
    out['body_x_vol']          = out['body_abs'] * out['VOL']
    out['signed_vol']          = out['direction'] * out['VOL']
    out['money_flow_proxy']    = ((out['CLOSE'] - out['LOW']) - (out['HIGH'] - out['CLOSE'])) / safe_rng * out['VOL']
    return out


# ── 3. Rolling context features ───────────────────────────────────────────
def add_rolling_features(df: pd.DataFrame, windows=(6, 12, 24)) -> pd.DataFrame:
    '''
    Rolling mean/std/zscore по close, range, body, volume.
    min_periods=w гарантирует NaN на разогреве — защита от bias.
    '''
    out = df.copy(deep=True)
    for w in windows:
        out[f'ret_{w}']           = out['CLOSE'].pct_change(w)
        rng_mean = out['candle_range'].rolling(w, min_periods=w).mean()
        rng_std  = out['candle_range'].rolling(w, min_periods=w).std()
        out[f'range_mean_{w}']    = rng_mean
        out[f'range_std_{w}']     = rng_std
        out[f'range_zscore_{w}']  = (out['candle_range'] - rng_mean) / rng_std.replace(0, np.nan)
        out[f'body_mean_{w}']     = out['body_abs'].rolling(w, min_periods=w).mean()
        close_ma  = out['CLOSE'].rolling(w, min_periods=w).mean()
        close_std = out['CLOSE'].rolling(w, min_periods=w).std()
        out[f'close_ma_{w}']      = close_ma
        out[f'close_vs_ma_{w}']   = out['CLOSE'] / close_ma - 1.0
        out[f'close_zscore_{w}']  = (out['CLOSE'] - close_ma) / close_std.replace(0, np.nan)
        vol_ma = out['VOL'].rolling(w, min_periods=w).mean()
        out[f'vol_ma_{w}']        = vol_ma
        out[f'vol_ratio_{w}']     = out['VOL'] / vol_ma.replace(0, np.nan)
    return out


# ── 4. Calendar / session features ───────────────────────────────────────
def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Признаки времени суток, дня недели, позиции бара в сессии.
    Требует колонку 'datetime' (добавляется в prepare_ohlcv_dataframe).
    Циклические признаки hour_sin/cos, dow_sin/cos — для моделей без понятия о порядке.
    '''
    if 'datetime' not in df.columns:
        raise ValueError("Column 'datetime' not found. Run prepare_ohlcv_dataframe first.")
    out = df.copy(deep=True)
    dt = out['datetime']
    out['hour']        = dt.dt.hour
    out['minute']      = dt.dt.minute
    out['dayofweek']   = dt.dt.dayofweek
    out['hour_sin']    = np.sin(2 * np.pi * out['hour'] / 24)
    out['hour_cos']    = np.cos(2 * np.pi * out['hour'] / 24)
    out['dow_sin']     = np.sin(2 * np.pi * out['dayofweek'] / 5)
    out['dow_cos']     = np.cos(2 * np.pi * out['dayofweek'] / 5)
    out['minutes_from_session_open'] = out['hour'] * 60 + out['minute'] - 600
    out['is_opening_30m']  = (out['minutes_from_session_open'] < 30).astype('int8')
    out['is_first_hour']   = (out['minutes_from_session_open'] < 60).astype('int8')
    out['date_only'] = dt.dt.date
    out['bar_in_day'] = out.groupby('date_only').cumcount()
    out['is_first_bar_of_day'] = (out['bar_in_day'] == 0).astype('int8')
    out = out.drop(columns=['date_only'])
    return out


# ── 5. Технические индикаторы (pandas_ta) ─────────────────────────────────
def add_ta_features(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Добавляет технические индикаторы через pandas_ta.
    Все индикаторы вычисляются только по текущим и прошлым барам.
    '''
    out = df.copy(deep=True)
    # EMA / SMA
    out['ema_10']    = ta.ema(out['CLOSE'], length=10)
    out['ema_20']    = ta.ema(out['CLOSE'], length=20)
    out['sma_20']    = ta.sma(out['CLOSE'], length=20)
    # RSI (momentum oscillator)
    out['rsi_14']    = ta.rsi(out['CLOSE'], length=14)
    # ATR (volatility)
    out['atr_14']    = ta.atr(out['HIGH'], out['LOW'], out['CLOSE'], length=14)
    out['natr_14']   = ta.natr(out['HIGH'], out['LOW'], out['CLOSE'], length=14)
    # MACD
    macd = ta.macd(out['CLOSE'], fast=12, slow=26, signal=9)
    if macd is not None:
        out = pd.concat([out, macd], axis=1)
    # Bollinger Bands
    bbands = ta.bbands(out['CLOSE'], length=20)
    if bbands is not None:
        out = pd.concat([out, bbands], axis=1)
        # Позиция close внутри полос Боллинджера [0, 1]
        bbl = f'BBL_20_2.0'
        bbu = f'BBU_20_2.0'
        if bbl in out.columns and bbu in out.columns:
            bb_rng = (out[bbu] - out[bbl]).replace(0, np.nan)
            out['close_pos_in_bbands'] = (out['CLOSE'] - out[bbl]) / bb_rng
    # Stochastic
    stoch = ta.stoch(out['HIGH'], out['LOW'], out['CLOSE'])
    if stoch is not None:
        out = pd.concat([out, stoch], axis=1)
    # OBV (on-balance volume)
    out['obv'] = ta.obv(out['CLOSE'], out['VOL'])
    # Price vs moving averages
    out['price_vs_ema_10'] = out['CLOSE'] / out['ema_10'] - 1.0
    out['price_vs_ema_20'] = out['CLOSE'] / out['ema_20'] - 1.0
    out['price_vs_sma_20'] = out['CLOSE'] / out['sma_20'] - 1.0
    return out


# ── 6. Target variable ────────────────────────────────────────────────────
def add_target(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Добавляет бинарный target: будет ли следующая свеча зелёной.
    target_is_green_next = 1, если CLOSE[t+1] > OPEN[t+1], иначе 0.
    ВАЖНО: сдвиг вперёд (shift(-1)) — это look-ahead ТОЛЬКО для target,
           что допустимо. Фичи target не видят.
    '''
    out = df.copy(deep=True)
    out['next_open']  = out['OPEN'].shift(-1)
    out['next_close'] = out['CLOSE'].shift(-1)
    out['target_is_green_next'] = (out['next_close'] > out['next_open']).astype('int8')
    out = out.drop(columns=['next_open', 'next_close'])
    # Последняя строка не имеет следующей свечи — убираем
    out = out.iloc[:-1].reset_index(drop=True)
    return out

print('Feature engineering functions defined ✓')

## Cell 4 — Master Pipeline Builder

In [ ]:
def build_feature_dataframe(df_raw: pd.DataFrame) -> pd.DataFrame:
    '''
    Полный пайплайн подготовки признаков:
    1. Подготовка OHLCV (типы, datetime, сортировка)
    2. Геометрия свечи + однобарные returns
    3. Rolling context (волатильность, объём, close vs MA)
    4. Calendar / session признаки
    5. Технические индикаторы (pandas_ta)
    6. Целевая переменная (target_is_green_next)
    '''
    df = prepare_ohlcv_dataframe(df_raw)
    df = add_domain_features(df)
    df = add_rolling_features(df, windows=(6, 12, 24))
    df = add_calendar_features(df)
    df = add_ta_features(df)
    df = add_target(df)
    return df


def get_leakage_columns(df: pd.DataFrame) -> list:
    '''
    Возвращает список колонок, которые нельзя использовать как фичи модели:
    - target-колонки
    - datetime и мета-информация
    - OHLCV-сырые данные (они содержат информацию о текущем баре,
      которую модель не должна получать напрямую — только через фичи)
    '''
    leakage = [
        'datetime', 'DATE', 'TIME', 'TICKER', 'PER',
        'OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOL',  # сырые OHLCV
    ]
    # Автоматически добавляем все target-колонки
    target_cols = [c for c in df.columns if c.startswith('target_')]
    return list(set(leakage + target_cols))


def build_X_y_for_model(feature_df: pd.DataFrame, n_in: int = 5):
    '''
    Строит X (лаговые признаки через series_to_supervised) и y (target).
    
    Parameters:
    ----------
    feature_df : pd.DataFrame — датафрейм с фичами и target.
    n_in : int — количество лаговых баров (по умолчанию 5 = 25 минут).
    
    Returns:
    -------
    X_supervised : pd.DataFrame
    y_aligned : pd.Series
    leakage_cols : list
    '''
    leakage_cols = get_leakage_columns(feature_df)
    # Оставляем только фичи (без leakage-колонок и без target)
    feature_only_cols = [c for c in feature_df.columns if c not in leakage_cols]
    X_base = feature_df[feature_only_cols].copy()
    y_full = feature_df['target_is_green_next'].copy()
    # Удаляем строки, где любая фича — NaN (warm-up от rolling)
    valid_idx = X_base.dropna().index
    X_clean = X_base.loc[valid_idx].reset_index(drop=True)
    y_clean = y_full.loc[valid_idx].reset_index(drop=True)
    # Применяем series_to_supervised — добавляем n_in лаговых баров
    X_supervised = series_to_supervised(X_clean, n_in=n_in, n_out=1, dropnan=True)
    # Выравниваем y по индексу X_supervised
    y_aligned = y_clean.iloc[n_in:].reset_index(drop=True)
    assert len(X_supervised) == len(y_aligned), 'X/y length mismatch after supervised framing'
    return X_supervised, y_aligned, leakage_cols

print('Pipeline builder functions defined ✓')

## Cell 5 — Load Data & Build Features

In [ ]:
# ── Загрузка данных ───────────────────────────────────────────────────────
# Данные скачаны с https://www.finam.ru/profile/moex-akcii/sberbank/export/
# Формат: TICKER;PER;DATE;TIME;OPEN;HIGH;LOW;CLOSE;VOL

DATA_PATH = './data/Сбербанк/year_result.csv'

frame = pd.read_csv(DATA_PATH, header=0, sep=';')

# Убираем угловые скобки из имён колонок Finam (<DATE> → DATE)
frame.columns = [c.strip('<>').strip() for c in frame.columns]

print(f'Загружено строк: {len(frame):,}')
print(f'Колонки: {frame.columns.tolist()}')
print(f'Диапазон дат: {frame["DATE"].min()} — {frame["DATE"].max()}')
frame.head(3)

In [ ]:
# ── Запуск полного пайплайна feature engineering ──────────────────────────
feature_df = build_feature_dataframe(frame)

print(f'Итоговый DataFrame: {feature_df.shape[0]:,} строк × {feature_df.shape[1]} колонок')
print(f'Баланс целевой переменной:')
print(feature_df['target_is_green_next'].value_counts(normalize=True).rename({0: 'red (0)', 1: 'green (1)'}).round(3))
feature_df.head(3)

## Cell 6 — EDA: Distribution of Key Features

In [ ]:
# ── EDA — распределение ключевых признаков ────────────────────────────────
eda_cols = ['ret_1', 'body_to_range', 'close_pos_in_range', 'range_pct_close',
            'vol_ratio_6', 'rsi_14', 'close_vs_ma_12']

fig, axes = plt.subplots(1, len(eda_cols), figsize=(20, 4))
for ax, col in zip(axes, eda_cols):
    if col in feature_df.columns:
        feature_df[col].dropna().hist(bins=60, ax=ax, color='steelblue', edgecolor='none', alpha=0.8)
        ax.set_title(col, fontsize=9)
        ax.set_ylabel('')
plt.suptitle('Feature distributions (raw)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Корреляционная матрица топ-признаков ─────────────────────────────────
top_feat = ['ret_1', 'ret_6', 'ret_12', 'body_to_range', 'close_pos_in_range',
            'vol_ratio_6', 'rsi_14', 'close_vs_ma_12', 'range_zscore_12',
            'is_opening_30m', 'direction', 'target_is_green_next']
top_feat = [c for c in top_feat if c in feature_df.columns]

plt.figure(figsize=(12, 9))
corr = feature_df[top_feat].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix — Key Features + Target')
plt.tight_layout()
plt.show()

## Cell 7 — Build Supervised Dataset & Time Split

In [ ]:
# ── Supervised framing (n_in=5 лаговых баров = 25 минут истории) ──────────
X, y, leakage_cols = build_X_y_for_model(feature_df, n_in=5)
print(f'X shape: {X.shape}  |  y shape: {y.shape}')
print(f'Баланс классов в y: {y.value_counts(normalize=True).round(3).to_dict()}')

# ── Хронологический split: train 70% / valid 15% / calibration 15% ────────
# НИКАКОГО перемешивания — временные ряды нельзя перемешивать.
# Это самый важный методологический момент при работе с финансовыми данными.
n = len(X)
n_train = int(n * 0.70)
n_valid = int(n * 0.15)
# Всё оставшееся идёт на калибровку
X_train, y_train = X.iloc[:n_train], y.iloc[:n_train]
X_valid, y_valid = X.iloc[n_train:n_train+n_valid], y.iloc[n_train:n_train+n_valid]
X_calib, y_calib = X.iloc[n_train+n_valid:], y.iloc[n_train+n_valid:]

for name, Xs, ys in [('Train', X_train, y_train), ('Valid', X_valid, y_valid), ('Calib', X_calib, y_calib)]:
    bal = ys.value_counts(normalize=True).round(3).to_dict()
    print(f'{name:6s}: {len(Xs):6,} rows | class balance: {bal}')

## Cell 8 — Baseline Model: Logistic Regression

In [ ]:
# ── Вспомогательная функция оценки модели ────────────────────────────────
def evaluate_model(model, X_tr, y_tr, X_va, y_va, X_ca, y_ca, model_name='Model'):
    '''Печатает метрики на трёх сегментах и возвращает словарь с результатами.'''
    results = {}
    for split_name, Xs, ys in [('Train', X_tr, y_tr), ('Valid', X_va, y_va), ('Calib', X_ca, y_ca)]:
        preds  = model.predict(Xs)
        probas = model.predict_proba(Xs)[:, 1]
        acc  = accuracy_score(ys, preds)
        auc  = roc_auc_score(ys, probas)
        ll   = log_loss(ys, probas)
        brier = brier_score_loss(ys, probas)
        results[split_name] = {'accuracy': acc, 'roc_auc': auc, 'log_loss': ll, 'brier': brier}
        print(f'[{model_name}] {split_name:6s} | Acc={acc:.4f} AUC={auc:.4f} LogLoss={ll:.4f} Brier={brier:.4f}')
    return results


# ── Baseline: LogisticRegression в Pipeline со StandardScaler ─────────────
# class_weight='balanced' компенсирует дисбаланс классов автоматически.
baseline_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
])

# Обучаем ТОЛЬКО на train — не трогаем valid и calib до оценки!
baseline_model.fit(X_train, y_train)
print('=== Baseline: Logistic Regression ===')
baseline_results = evaluate_model(baseline_model, X_train, y_train, X_valid, y_valid, X_calib, y_calib, 'LR')

In [ ]:
# ── Classification Report и Confusion Matrix на Valid ─────────────────────
print('\n=== Classification Report (Valid) ===')
print(classification_report(y_valid, baseline_model.predict(X_valid), target_names=['red (0)', 'green (1)']))

plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_valid, baseline_model.predict(X_valid))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Red', 'Pred Green'],
            yticklabels=['Actual Red', 'Actual Green'])
plt.title('Confusion Matrix — Logistic Regression (Valid)')
plt.tight_layout()
plt.show()

## Cell 9 — Probability Calibration (Platt Scaling)

In [ ]:
# ── Калибровка вероятностей (Platt / sigmoid) ─────────────────────────────
# Uncalibrated классификатор может давать хороший AUC, но плохие вероятности.
# CalibratedClassifierCV(cv='prefit') обучает сигмоиду поверх уже обученной модели
# на отдельном calibration holdout — это исключает переобучение калибровки.

calibrated_model = CalibratedClassifierCV(
    estimator=baseline_model,
    method='sigmoid',   # Platt scaling
    cv='prefit',        # baseline_model уже обучен, используем его как есть
)
calibrated_model.fit(X_calib, y_calib)  # Обучаем калибровщик на calib-сегменте

print('=== Calibrated Model ===')
calib_results = evaluate_model(calibrated_model, X_train, y_train, X_valid, y_valid, X_calib, y_calib, 'LR+Calib')

In [ ]:
# ── Calibration Curve (Reliability Diagram) ───────────────────────────────
# Идеально откалиброванная модель: predicted probability == actual frequency.
# На графике — прямая диагональ.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Calibration curves ---
ax = axes[0]
for model, label, color in [
    (baseline_model, 'Baseline LR', 'steelblue'),
    (calibrated_model, 'LR + Platt', 'tomato'),
]:
    prob_true, prob_pred = calibration_curve(y_valid, model.predict_proba(X_valid)[:, 1], n_bins=10)
    ax.plot(prob_pred, prob_true, marker='o', label=label, color=color)
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration Curve (Valid)')
ax.legend()

# --- Probability histograms ---
ax2 = axes[1]
ax2.hist(baseline_model.predict_proba(X_valid)[:, 1], bins=40,
         alpha=0.6, label='Baseline LR', color='steelblue', edgecolor='none')
ax2.hist(calibrated_model.predict_proba(X_valid)[:, 1], bins=40,
         alpha=0.6, label='LR + Platt', color='tomato', edgecolor='none')
ax2.set_xlabel('Predicted probability (class=1, green)')
ax2.set_ylabel('Count')
ax2.set_title('Probability Distribution (Valid)')
ax2.legend()

plt.tight_layout()
plt.show()

## Cell 10 — Metrics Summary Table

In [ ]:
# ── Сводная таблица метрик: Baseline vs Calibrated ───────────────────────
rows = []
for split in ['Train', 'Valid', 'Calib']:
    r_base  = baseline_results[split]
    r_calib = calib_results[split]
    rows.append({
        'Split':          split,
        'LR Accuracy':    round(r_base['accuracy'], 4),
        'LR AUC':         round(r_base['roc_auc'],  4),
        'LR LogLoss':     round(r_base['log_loss'], 4),
        'LR Brier':       round(r_base['brier'],    4),
        'Cal Accuracy':   round(r_calib['accuracy'],4),
        'Cal AUC':        round(r_calib['roc_auc'], 4),
        'Cal LogLoss':    round(r_calib['log_loss'],4),
        'Cal Brier':      round(r_calib['brier'],   4),
    })
metrics_df = pd.DataFrame(rows).set_index('Split')
print('\n=== Metrics Summary: Logistic Regression vs Calibrated ===')
metrics_df

## Next Steps

- [ ] Walk-forward validation (expanding window)
- [ ] GradientBoostingClassifier vs LogisticRegression — сравнение по тем же метрикам
- [ ] Feature importance (permutation importance)
- [ ] Симуляция транзакционных издержек
- [ ] Regime detection (rolling volatility / HMM)
- [ ] Meta-labeling для фильтрации сигналов